# 06 · Transfer learning: reuse a pretrained vision model

**Level:** intermediate. **Before this:** understand a CNN training loop (notebooks 03–05).
**Practical outcome:** adapt ResNet-18 to ten CIFAR-10 classes, first train its classifier,
then fine-tune its last residual block, and select a model using validation data.

Run each cell in order. Every notebook can start from a fresh kernel. `QUICK=True` uses
small teaching subsets, so results are demonstrations, not benchmark scores. An internet
connection is needed for the first CIFAR-10 and pretrained-weight download. On an 8 GB
GPU, start with batch size 16; reduce it to 8 if another app occupies GPU memory.


## Step 1 · Configure the session

The setup prints the active device. CUDA uses your NVIDIA GPU. CPU fallback works but
224-pixel images take longer. Keep `QUICK=True` for a live demonstration; use `False`
for a larger practice run. `COURSE_SMOKE=1` is a tiny pipeline check, not an accuracy run.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


In [ ]:
import copy
import pandas as pd
from torchvision import models, transforms
from torch import nn

cu.seed_everything(42)
BATCH_SIZE = 16
HEAD_EPOCHS = 1 if SMOKE else (2 if QUICK else 5)
FINE_TUNE_EPOCHS = 1 if SMOKE else (2 if QUICK else 5)
limits = (32, 16, 16) if SMOKE else ((800, 200, 300) if QUICK else (8000, 1500, 2000))
print("Dataset limits:", limits, "| Epochs:", HEAD_EPOCHS, "+", FINE_TUNE_EPOCHS)


## Step 2 · Use the preprocessing expected by the weights

ImageNet-pretrained ResNet expects RGB inputs with its training normalization.
The weight object's `transforms()` supplies deterministic validation/inference
preprocessing. Training uses a random crop and horizontal flip with the same mean
and standard deviation. CIFAR-10 starts at 32 × 32; resizing provides the expected
input shape, but cannot recover detail that was never in the original image.


In [ ]:
weights = models.ResNet18_Weights.DEFAULT
eval_transform = weights.transforms()
MEAN = list(eval_transform.mean)
STD = list(eval_transform.std)
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
print("Pretrained weights:", weights)
print("Inference preprocessing:", eval_transform)


## Step 3 · Split before comparing models

Training and validation are disjoint subsets of CIFAR-10's training partition.
The official test partition stays out of the model-selection loop. Training gets
random augmentation; validation and test get the deterministic weight preprocessing.


In [ ]:
train_ds, val_ds, test_ds, class_names = cu.cifar_data(
    train_transform=train_transform, eval_transform=eval_transform,
    train_limit=limits[0], val_limit=limits[1], test_limit=limits[2], seed=42,
)
train_dl = cu.loader(train_ds, batch_size=BATCH_SIZE, shuffle=True, seed=42)
val_dl = cu.loader(val_ds, batch_size=BATCH_SIZE)
test_dl = cu.loader(test_ds, batch_size=BATCH_SIZE)
print("Train / validation / test:", len(train_ds), len(val_ds), len(test_ds))
cu.show_images(val_ds, class_names, n=8, mean=MEAN, std=STD)


## Step 4 · Inspect the network and replace its classifier

The original classifier produces 1,000 ImageNet logits. Our replacement produces
ten CIFAR-10 logits. A logit is an unnormalized class score; cross-entropy expects
these raw scores. The rest of the network supplies reusable image features.
The first model construction downloads the pretrained weights into PyTorch's cache.


In [ ]:
model = models.resnet18(weights=weights)
for parameter in model.parameters():
    parameter.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(DEVICE)

def parameter_counts(network):
    return {
        "total": sum(p.numel() for p in network.parameters()),
        "trainable": sum(p.numel() for p in network.parameters() if p.requires_grad),
    }

print(model.fc)
print(parameter_counts(model))
xb, yb = next(iter(val_dl))
model.eval()
with torch.inference_mode():
    print("Input:", tuple(xb.shape), "→ logits:", tuple(model(xb.to(DEVICE)).shape))


## Step 5 · Freeze behavior as well as gradients

`requires_grad=False` prevents weight gradients; it does **not** stop BatchNorm
running-statistic updates. `model.train()` normally enables those updates. Our loop
holds every BatchNorm layer in evaluation mode during both phases. This is a
deliberate policy for a small dataset; the classifier and selected convolution
weights can still learn. `eval()` does not turn gradient computation off.


In [ ]:
def hold_batchnorm_statistics(network):
    for module in network.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()

def transfer_epoch(network, dataloader, optimizer):
    network.train()
    hold_batchnorm_statistics(network)
    total_loss, correct, count = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = network(images)
        loss = nn.functional.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        n = labels.numel()
        total_loss += loss.item() * n
        correct += (logits.argmax(1) == labels).sum().item()
        count += n
    return {"loss": total_loss / count, "accuracy": correct / count}

def run_phase(network, optimizer, epochs, title):
    best_loss = float("inf")
    best_state = None
    history = []
    for epoch in range(1, epochs + 1):
        train_metrics = transfer_epoch(network, train_dl, optimizer)
        val_metrics = cu.evaluate(network, val_dl, DEVICE)
        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"], "val_loss": val_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "val_accuracy": val_metrics["accuracy"],
        })
        print(f"{title} {epoch}/{epochs}: val_loss={val_metrics['loss']:.4f}, "
              f"val_accuracy={val_metrics['accuracy']:.3f}")
        if val_metrics["loss"] < best_loss:
            best_loss = val_metrics["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in network.state_dict().items()}
    network.load_state_dict(best_state)
    return history, best_state


## Step 6 · Train only the new head

Only the classifier parameters enter the optimizer. Inspect the trainable parameter
count and BatchNorm check below while explaining why this phase is inexpensive.
We retain the epoch with the lowest validation loss, which may be earlier than the
last epoch. Do not choose settings from test accuracy.


In [ ]:
head_optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)
bn_before = model.bn1.running_mean.detach().clone()
head_history, head_state = run_phase(model, head_optimizer, HEAD_EPOCHS, "Frozen backbone")
assert torch.equal(bn_before, model.bn1.running_mean), "Frozen BatchNorm statistics changed."
head_validation = cu.evaluate(model, val_dl, DEVICE)
cu.plot_history(head_history)


## Step 7 · Fine-tune the last residual block

Continue from the best head-only state. Unfreeze `layer4`, keep BatchNorm affine
parameters and running statistics frozen, and use a smaller learning rate for the
pretrained convolutions. The new classifier gets a somewhat larger rate. Recreate
the optimizer so the newly trainable parameters are included.


In [ ]:
for parameter in model.layer4.parameters():
    parameter.requires_grad = True
for module in model.modules():
    if isinstance(module, nn.modules.batchnorm._BatchNorm):
        for parameter in module.parameters():
            parameter.requires_grad = False

fine_optimizer = torch.optim.AdamW([
    {"params": [p for p in model.layer4.parameters() if p.requires_grad], "lr": 1e-5},
    {"params": model.fc.parameters(), "lr": 1e-4},
], weight_decay=1e-4)
print(parameter_counts(model))
fine_history, fine_state = run_phase(model, fine_optimizer, FINE_TUNE_EPOCHS, "Fine-tune layer4")
fine_validation = cu.evaluate(model, val_dl, DEVICE)
cu.plot_history(fine_history)


## Step 8 · Compare on validation and select once

Fine-tuning does not guarantee improvement, especially with a very small training
subset. The rule below selects the lower validation loss, using the frozen-backbone
model on a tie. This is a sequential experiment: fine-tuning starts from phase one's
best state, rather than being an independent from-scratch trial.


In [ ]:
comparison = pd.DataFrame([
    {"phase": "head_only", **head_validation},
    {"phase": "fine_tuned", **fine_validation},
]).set_index("phase")
display(comparison)
selected_phase = "fine_tuned" if fine_validation["loss"] < head_validation["loss"] else "head_only"
model.load_state_dict(fine_state if selected_phase == "fine_tuned" else head_state)
print("Selected using validation loss:", selected_phase)
comparison["accuracy"].plot.bar(ylim=(0, 1), title="Validation accuracy (selection uses loss)")
plt.ylabel("Accuracy")
plt.xticks(rotation=0)
plt.show()


## Step 9 · Open the held-out test set

The experiment is now fixed. Evaluate the chosen model once, then inspect several
predictions. Softmax scores rank the model's classes; they are not a guarantee of
calibrated real-world confidence. Repeatedly tuning after viewing this result would
turn the test set into another validation set.


In [ ]:
test_metrics = cu.evaluate(model, test_dl, DEVICE)
print({"selected_phase": selected_phase, "test": test_metrics})
images, labels = next(iter(test_dl))
model.eval()
with torch.inference_mode():
    probabilities = model(images.to(DEVICE)).softmax(1).cpu()
fig, axes = plt.subplots(1, min(6, len(images)), figsize=(15, 3))
for i, ax in enumerate(np.atleast_1d(axes)):
    rgb = images[i] * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]
    predicted = probabilities[i].argmax().item()
    ax.imshow(rgb.permute(1, 2, 0).clamp(0, 1))
    ax.set_title(f"True: {class_names[labels[i].item()]}\nPred: {class_names[predicted]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Step 10 · Save the model together with its meaning

Weights alone do not record the class order or input transformation. Save these
alongside the architecture and selection information. The reload example constructs
the network without downloading pretrained weights again: our saved state already
contains all its parameters and buffers.


In [ ]:
checkpoint_path = cu.ARTIFACTS / "06_resnet18_cifar10.pt"
torch.save({
    "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "architecture": "resnet18", "class_names": list(class_names),
    "mean": MEAN, "std": STD, "resize": 256, "center_crop": 224,
    "selected_phase": selected_phase,
}, checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
restored = models.resnet18(weights=None)
restored.fc = nn.Linear(restored.fc.in_features, len(checkpoint["class_names"]))
restored.load_state_dict(checkpoint["model_state"])
restored = restored.to(DEVICE).eval()
with torch.inference_mode():
    original_logits = model.eval()(images[:2].to(DEVICE))
    restored_logits = restored(images[:2].to(DEVICE))
torch.testing.assert_close(original_logits, restored_logits)
print("Saved and verified:", checkpoint_path)


## Step 11 · Demonstrate an actual prediction on your own image

Set `IMAGE_PATH` to a local image to try this cell. Predictions are restricted to the
ten CIFAR-10 categories; a portrait or document still receives one of those labels.
Use the same deterministic preprocessing used for validation. The `None` default
keeps Run All independent of your personal files.


In [ ]:
from PIL import Image

IMAGE_PATH = None  # Example: Path(r"C:\Users\YourName\Pictures\dog.jpg")
if IMAGE_PATH is None:
    print("Optional: set IMAGE_PATH to classify a local image.")
else:
    with Image.open(IMAGE_PATH) as source:
        rgb_image = source.convert("RGB")
    with torch.inference_mode():
        scores = restored(eval_transform(rgb_image).unsqueeze(0).to(DEVICE)).softmax(1)[0].cpu()
    values, indices = scores.topk(3)
    display(pd.DataFrame({
        "class": [class_names[i] for i in indices.tolist()],
        "softmax_score": values.tolist(),
    }))
    plt.imshow(rgb_image)
    plt.axis("off")
    plt.show()


## Step 12 · Pause, explain, and experiment

1. Why can BatchNorm change when all its parameters have `requires_grad=False`?
2. Print the names of the trainable parameters in each phase.
3. In a **new validation experiment**, try fine-tuning `layer3` too. Compare validation
   loss and training time. Keep the existing test result out of this decision.
4. Explain what information belongs in an inference artifact besides weights.

<details><summary>Hints / solutions</summary>

BatchNorm's running mean and variance are buffers, not gradient-updated parameters.
Use evaluation mode to hold those statistics fixed. For inspection:

```python
[name for name, p in model.named_parameters() if p.requires_grad]
```

To include `layer3`, unfreeze its convolution weights, keep BatchNorm frozen, and add
those parameters to a **new** optimizer with a small learning rate. Save architecture,
class order, normalization, resize/crop policy, and the selected trained state.
Choose final settings on validation, then use a fresh held-out set for a new final claim.

</details>

**References:** [PyTorch transfer learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html),
[ResNet-18 weights and preprocessing](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html),
[BatchNorm behavior](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html).
